In [ ]:
# # ── TEMPORARY: Suppress all Plotly charts to prevent VS Code freeze ──
# # Run this cell first, then run the rest of the notebook.
# # Delete or skip this cell when you want charts to render again.
# import plotly.graph_objects as go
# import plotly.express as px
# _orig_fig_show   = go.Figure.show
# go.Figure.show   = lambda *a, **kw: None   # no-op
# _orig_subplots = None  # subplots figures also use go.Figure.show — already covered
# print("⚠️  Chart rendering SUPPRESSED — text outputs only. Delete this cell to re-enable charts.")


# Pricing Strategy — Consolidated Analysis

**Objective:** Brand building via loyal user growth (Trial / Repeat / Lapse): define each size's most effective price point by understanding shopper flow.

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | **Analysis Brief** | Objective, definitions, parameters, analysis flow |
| 2 | **Setup & Data Pull** | Imports, parameters, SQL queries with caching |
| A | **Pane A — Total Shopper & ASP Landscape** | Market sizing, pricing trends, renewal impact |
| B | **Pane B — Trial Shopper** | Trial acquisition by size, ASP elasticity, price gap |
| C | **Pane C — Repeat Shopper** | Cohort funnel, size migration, ASP band analysis |
| D | **Pane D — Lapsed Shopper** | Lapse rate by size/ASP, post-lapse destination |
| E | **Pane E — Shopper Flow** | Sankey, ASP-annotated funnel, strategic map |
| F | **Pane F — Strategic Summary** | Synthesized findings & recommendations |

**Created:** 2026-02-19 | **Consolidated:** 2026-02-27


---
### 📏 Canonical Definitions

| Term | Definition | Window |
|------|-----------|--------|
| **Trial Shopper** | A shopper who purchases the sub-brand/category with **no purchase history of that sub-brand/category in the prior 12 months** (365 days). Rolling lookback per purchase event, NOT first-ever. | 365-day lookback |
| **Repeat Shopper** | A trial shopper who makes ≥1 subsequent purchase of the **same sub-brand** within **180 days** (6 months) after their trial event. | 180-day forward window |
| **Lapsed Shopper** | A trial shopper who makes **no subsequent purchase** of the same sub-brand within **180 days** (6 months) after their trial event. | 180-day forward window |
| **ASP** | `SUM(pos_sales_amt) / SUM(pos_unit_sales_qty)` — weighted average selling price. | Per transaction/aggregation |
| **ASP Band (50 JPY bin)** | `FLOOR(ASP / 50) * 50` — ASP floored to nearest 50 JPY. | — |

**Repeat + Lapse are mutually exclusive and exhaustive** within the trial cohort.

### Analysis Flow
```
Total Shoppers (ASP Landscape)
  |── Non Trial shopper (Repeat from prev period)
  └── Trial Shoppers (new-to-brand)
        ├── Repeat Shoppers (retained within 6 months)
        │     └── Size migration (entry size → repeat size)
        └── Lapsed Shoppers (no return within 6 months)
              └── Destination tracking (where did they go?)

At each stage: How does PRICE affect the shopper's behavior?
```

### Data Sources (IDPOS_REFERENCE.md)
- **Fact table:** `cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw`
- **Product dim:** `id_pos_ai_1.prod_dim_ext_vw`
- **Shopper dim:** `id_pos_ai_1.shopper_dim_generic_vw`
- **Partition keys:** `sales_period_group_end_date_part`, `data_provider_code_part`


---
# Section 2 — Setup, Parameters & Data Pull


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2-1. Imports & Connection
# ═══════════════════════════════════════════════════════════════════════
import os, time, warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
from dotenv import load_dotenv
import databricks.sql as sql

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

# ── Japanese font setup ────────────────────────────────────────────────
def _find_japanese_font():
    for name in ['MS Gothic', 'MS PGothic', 'Yu Gothic', 'Meiryo', 'IPAexGothic']:
        if name in {f.name for f in fm.fontManager.ttflist}:
            return name
    return None

_jp_font = _find_japanese_font()
if _jp_font:
    plt.rcParams['font.family'] = _jp_font
    print(f'✅ Japanese font: {_jp_font}')

# ── Databricks credentials ──────────────────────────────────────────────
load_dotenv(dotenv_path='../../.env')
DATABRICKS_HOST      = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN     = os.getenv('DATABRICKS_TOKEN')
DATABRICKS_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH')
assert all([DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_HTTP_PATH]), 'Missing .env credentials'
print('✅ Credentials loaded')

# ── Query helpers ────────────────────────────────────────────────────────
def execute_query(query: str) -> pd.DataFrame:
    with sql.connect(server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
                     access_token=DATABRICKS_TOKEN) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

def execute_query_long(query: str) -> pd.DataFrame:
    with sql.connect(server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
                     access_token=DATABRICKS_TOKEN,
                     _retry_stop_after_attempts_duration=3600) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

# ── Output helpers ───────────────────────────────────────────────────────
DATA_DIR   = Path('data')
OUTPUT_DIR = Path('output')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

def strip_tz(df):
    df = df.copy()
    for col in df.select_dtypes(include=['datetimetz']).columns:
        df[col] = df[col].dt.tz_localize(None)
    for col in df.columns:
        if df[col].dtype == 'object':
            try: df[col] = df[col].astype(str)
            except: pass
    return df

def fmt_km(v):
    if v >= 1_000_000: return f'{v/1_000_000:.1f}M'
    if v >= 1_000:     return f'{v/1_000:.1f}K'
    return str(int(v))

print('✅ Setup complete')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2-2. ALL Parameters (single source of truth)
# ═══════════════════════════════════════════════════════════════════════

# ── Target brands (half-width katakana per IDPOS_REFERENCE.md) ────────
Brand_1   = 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ'
Brand_2   = 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ'
SUB_CAT     = '洗濯洗剤'
CATEGORY    = 'Laundry'

# ── Time windows ──────────────────────────────────────────────────────
LOOKBACK_START = '2024-01-01'   # Start of LAG lookback window
ANALYSIS_START = '2025-01-01'   # Analysis period start
ANALYSIS_END   = '2026-01-31'   # Analysis period end
RENEWAL_MONTH  = '2025-05-01'   # Product renewal breakpoint

# ── Canonical shopper classification windows ──────────────────────────
TRIAL_LOOKBACK_DAYS       = 365  # Trial = no purchase in prior 365 days
REPEAT_LAPSE_WINDOW_DAYS  = 180  # Repeat/Lapse = 180 days post-trial
LATEST_COHORT_END         = '2025-07-31'  # Latest cohort with 6-month follow-up
LAPSE_CUTOFF_DATE         = '2025-07-31'  # Confirm lapse after this date

# ── Retailer codes (8 national retailers) ─────────────────────────────
RETAILER_CODES = [
    'cds_8005', 'cds_8006', 'cds_8007', 'cds_8008', 'cds_8009',
    'cds_8010', 'cds_8011', 'cds_8013',
]
RETAILER_IN = ', '.join(f"'{c}'" for c in RETAILER_CODES)

# # ── Size hierarchy (physical size: small → large) ─────────────────────
# To copilot please deleate the following lines and replace with query to get top 6 value size from sales data. The hierarchy should be ordered from smallest to largest size, and any sizes that are not in the top 6 should be excluded from the analysis. If you can't understand what size is big or small, please refer to ASP.:
# SIZE_ORDER     = ['本体通常', '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替ﾃﾗｼﾞｬﾝﾎﾞ']
# EXCLUDED_SIZES = ["詰替超ｼﾞｬﾝﾎﾞ","詰替超ﾃﾗｼﾞｬﾝﾎﾞ","ｿﾉﾀ","詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ","詰替超特大","詰替通常","詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ","詰替特大"]

# SIZE_ROLE = {
#     '本体通常':          'Trial Entry',
#     '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ': 'Intermission', 
#     '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ':  'Loyalty', 
#     '詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ':  'Loyalty',
#     '詰替ﾃﾗｼﾞｬﾝﾎﾞ':  'Loyalty'}



# ── Frequency thresholds ──────────────────────────────────────────────
MIN_FREQ       = 100    # week X store frequency threshold

# ── Visualization colors ──────────────────────────────────────────────
BRAND_COLOR = {Brand_1: '#1E90FF', Brand_2: '#FF6347'}
BRAND_LABEL = {Brand_1: 'Bold Gel Ball', Brand_2: 'Ariel Gel Ball'}   # To copilot please change the lavel to Brand 1 value so that we can use the same lavel in the charts


def order_and_filter_sizes(sizes_list):
    sizes_set = set(sizes_list) - set(EXCLUDED_SIZES)
    return [s for s in SIZE_ORDER if s in sizes_set]

EXCLUDED_SIZES_SQL = ', '.join(f"'{s}'" for s in EXCLUDED_SIZES)

print('📋 Parameters loaded:')
print(f'  Analysis window : {ANALYSIS_START} → {ANALYSIS_END}')
print(f'  Lookback start  : {LOOKBACK_START}')
print(f'  Renewal month   : {RENEWAL_MONTH}')
print(f'  Cohort end      : {LATEST_COHORT_END}')
print(f'  Lapse cutoff    : {LAPSE_CUTOFF_DATE}')
print(f'  Retailers       : {len(RETAILER_CODES)}')
print(f'  Size order      : {SIZE_ORDER}')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 2-3. Smart Cache Layer
# ═══════════════════════════════════════════════════════════════════════
# Set to True after first successful run → all queries load from disk instantly.
# Set to False to force re-query from Databricks.

RELOAD_FROM_CACHE = True   # ← Toggle this!

_query_log = []

def cached_query(name: str, sql_str: str, force: bool = False, long: bool = False) -> pd.DataFrame:
    parquet_path = DATA_DIR / f'{name}.parquet'
    csv_path     = DATA_DIR / f'{name}.csv'

    # Check cache
    if RELOAD_FROM_CACHE and not force and parquet_path.exists():
        mod_time = datetime.fromtimestamp(parquet_path.stat().st_mtime)
        age_days = (datetime.now() - mod_time).days
        df = pd.read_parquet(parquet_path)
        status = f'📂 CACHE ({age_days}d old)'
        if age_days > 7:
            status += ' ⚠️ >7 days old'
        print(f'{status} | {name}: {len(df):,} rows | {parquet_path}')
        _query_log.append({'query': name, 'rows': len(df), 'seconds': 0, 'source': 'cache'})
        return df

    # Run query
    print(f'⏳ Querying Databricks: {name}...', flush=True)
    t0 = time.time()
    df = execute_query_long(sql_str) if long else execute_query(sql_str)
    elapsed = time.time() - t0

    # Save cache
    df.to_parquet(parquet_path, index=False)
    df.to_csv(csv_path, index=False)
    print(f'✅ {name}: {len(df):,} rows | {elapsed:.1f}s | saved to {parquet_path}')
    _query_log.append({'query': name, 'rows': len(df), 'seconds': round(elapsed, 1), 'source': 'databricks'})
    return df

print(f'Cache mode: {"RELOAD FROM CACHE" if RELOAD_FROM_CACHE else "QUERY DATABRICKS"}')
print(f'Cache dir : {DATA_DIR.resolve()}')


---
## Data Pull — 8 Active Queries (reduced from 12)

| # | Query Name | Purpose | Status |
|---|-----------|---------|--------|
| Q1 | `monthly_sales` | Monthly sales/ASP/shoppers by brand × size | ✅ Active |
| ~~Q2~~ | ~~`asp_dq_check`~~ | ~~ASP data quality~~ | 💤 Commented out |
| Q3 | `product_metadata` | Product attributes for both brands | ✅ Active |
| Q4 | `trial_cohort_full` | **Unified** trial→repeat/lapse at shopper level | ✅ Active (heaviest) |
| ~~Q5~~ | ~~`category_trial`~~ | ~~Category-level trial~~ | 💤 Commented out |
| Q6 | `asp_weekly_retailer` | Weekly × retailer × store count (absorbs Q11) | ✅ Active |
| Q7 | `asp_band_universe` | All shoppers per (size, ASP band) | ✅ Active |
| Q8 | `dual_brand_all_shoppers` | **All** shoppers + `is_lapsed` flag (absorbs Q10) | ✅ Active |
| Q9 | `dual_brand_active` | Active shoppers per size, both brands | ✅ Active |
| ~~Q10~~ | ~~`at_risk_asp`~~ | ~~At-risk shoppers~~ | ♻️ Derived from Q8 |
| ~~Q11~~ | ~~`week_store_coverage`~~ | ~~Week×store count~~ | ♻️ Derived from Q6 |
| Q12 | `dual_brand_destination` | Post-lapse destination for both brands | ✅ Active |


In [ ]:
# ──: Unified Trial → Repeat/Lapse Cohort (THE BIG ONE) ────────────
# Replaces NB02 subbrand_trial + NB03 cohort + NB05 journey
# Returns shopper-level trial→outcome for BOTH brands
q1_sql = f"""
WITH all_purchases AS (
    SELECT
        idpos.shopper_key,
        prod.jp_sub_brand_alter_lang_name AS sub_brand,
        prod.jp_segment_4_name            AS size_code,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS asp
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
           ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
           ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{LOOKBACK_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name IN ('{Brand_1}', '{Brand_2}')
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2, 3, 4
),
with_prev AS (
    SELECT *,
        LAG(purchase_date) OVER (
            PARTITION BY shopper_key, sub_brand ORDER BY purchase_date
        ) AS prev_purchase_date
    FROM all_purchases
),
trial_candidates AS (
    SELECT *
    FROM with_prev
    WHERE purchase_date BETWEEN '{ANALYSIS_START}' AND '{LATEST_COHORT_END}'
      AND (prev_purchase_date IS NULL
           OR DATEDIFF(purchase_date, prev_purchase_date) > {TRIAL_LOOKBACK_DAYS})
),
trial_events AS (
    SELECT shopper_key, sub_brand, MIN(purchase_date) AS trial_date
    FROM trial_candidates
    GROUP BY 1, 2
),
trial_detail AS (
    SELECT te.shopper_key, te.sub_brand, te.trial_date,
           tc.size_code AS trial_size, tc.asp AS trial_asp
    FROM trial_events te
    INNER JOIN trial_candidates tc
           ON te.shopper_key = tc.shopper_key
          AND te.sub_brand   = tc.sub_brand
          AND te.trial_date  = tc.purchase_date
),
repeat_events AS (
    SELECT
        td.shopper_key, td.sub_brand, td.trial_date, td.trial_size, td.trial_asp,
        MIN(ap.purchase_date) AS repeat_date,
        MIN(ap.size_code)     AS repeat_size,
        MIN(ap.asp)           AS repeat_asp
    FROM trial_detail td
    LEFT JOIN all_purchases ap
           ON td.shopper_key = ap.shopper_key
          AND td.sub_brand   = ap.sub_brand
          AND ap.purchase_date > td.trial_date
          AND ap.purchase_date <= DATE_ADD(td.trial_date, {REPEAT_LAPSE_WINDOW_DAYS})
    GROUP BY 1, 2, 3, 4, 5
)
SELECT
    re.shopper_key, re.sub_brand, re.trial_date, re.trial_size, re.trial_asp,
    re.repeat_date, re.repeat_size, re.repeat_asp,
    CASE WHEN re.repeat_date IS NOT NULL THEN 'Repeat' ELSE 'Lapse' END AS outcome,
    DATEDIFF(re.repeat_date, re.trial_date) AS days_to_repeat
FROM repeat_events re
ORDER BY re.sub_brand, re.trial_date
"""
df_cohort = cached_query('trial_cohort_full', q1_sql, long=True)
df_cohort['trial_date']  = pd.to_datetime(df_cohort['trial_date'])
df_cohort['repeat_date'] = pd.to_datetime(df_cohort['repeat_date'])
for col in ['trial_asp', 'repeat_asp', 'days_to_repeat']:
    df_cohort[col] = pd.to_numeric(df_cohort[col])
print(f'  Bold: {len(df_cohort[df_cohort["sub_brand"]==Brand_1]):,} | '
      f'Ariel: {len(df_cohort[df_cohort["sub_brand"]==Brand_2]):,}')


In [ ]:
# ── : Dual-Brand Post-Lapse Destination ────────────────────────────
q2_sql = f"""
WITH brand_last AS (
    SELECT idpos.shopper_key, prod.jp_sub_brand_alter_lang_name AS source_brand,
        MAX(CAST(idpos.sales_period_group_end_date_part AS DATE)) AS last_date
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    INNER JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    INNER JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}' AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name IN ('{Brand_1}', '{Brand_2}')
      AND shopper.member_ind = 'Y' AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2
    HAVING MAX(CAST(idpos.sales_period_group_end_date_part AS DATE)) <= DATE('{LAPSE_CUTOFF_DATE}')
),
next_laundry AS (
    SELECT bl.shopper_key, bl.source_brand,
        prod.jp_sub_brand_alter_lang_name AS next_sub_brand,
        prod.jp_segment_4_name            AS next_size,
        ROW_NUMBER() OVER (PARTITION BY bl.shopper_key, bl.source_brand
                           ORDER BY CAST(idpos.sales_period_group_end_date_part AS DATE)) AS rn
    FROM brand_last bl
    INNER JOIN cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
           ON bl.shopper_key = idpos.shopper_key
    INNER JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    WHERE CAST(idpos.sales_period_group_end_date_part AS DATE) > bl.last_date
      AND idpos.sales_period_group_end_date_part <= '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}' AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND idpos.pos_unit_sales_qty > 0
)
SELECT shopper_key, source_brand, next_sub_brand, next_size
FROM next_laundry WHERE rn = 1
"""
df_destination = cached_query('dual_brand_destination', q2_sql, long=True)
print(f'  Bold→dest: {len(df_destination[df_destination["source_brand"]==Brand_1]):,}')
print(f'  Ariel→dest: {len(df_destination[df_destination["source_brand"]==Brand_2]):,}')


In [ ]:
# ── B-3: Head-to-Head Sub-brand Trial by Size ─────────────────────────
h2h = df_trial[~df_trial['size_code'].isin(EXCLUDED_SIZES)].groupby(['sub_brand','size_code']).agg(
    subbrand_trial=('subbrand_trial_shoppers','sum'), avg_trial_rate=('trial_rate','mean')).reset_index()
h2h_sizes = order_and_filter_sizes(h2h['size_code'].unique())
h2h = h2h[h2h['size_code'].isin(h2h_sizes)].copy()

_ha = h2h[h2h['sub_brand']==Brand_1][['size_code','subbrand_trial','avg_trial_rate']].rename(
    columns={'subbrand_trial':'ariel_trial','avg_trial_rate':'ariel_rate'})
_hb = h2h[h2h['sub_brand']==Brand_2][['size_code','subbrand_trial','avg_trial_rate']].rename(
    columns={'subbrand_trial':'attack_trial','avg_trial_rate':'attack_rate'})
h2h_idx = _ha.merge(_hb, on='size_code', how='outer')
h2h_idx['index'] = (h2h_idx['ariel_trial'] / h2h_idx['attack_trial'].replace(0,np.nan) * 100).round(1)

print('📊 DATA TABLE: Head-to-Head Trial by Size')
print('=' * 80)
print(h2h_idx.to_string(index=False))

fig_h2h = make_subplots(specs=[[{'secondary_y':True}]])
for brand_code, brand_label, color in [(Brand_1, 'Bold Gel Ball', '#2980B9'), (Brand_2, 'Ariel Gel Ball', '#E67E22')]:
    d = h2h[h2h['sub_brand']==brand_code]
    fig_h2h.add_trace(go.Bar(x=d['size_code'], y=d['subbrand_trial'], name=brand_label,
        marker_color=color, opacity=0.8, text=[fmt_km(v) for v in d['subbrand_trial']], textposition='outside'),
        secondary_y=False)
fig_h2h.add_trace(go.Scatter(x=h2h_idx['size_code'], y=h2h_idx['index'], name='Index (Bold/Attack×100)',
    mode='lines+markers+text', line=dict(color='#27AE60', dash='dash', width=2.5),
    marker=dict(size=10, symbol='diamond', color='#27AE60'),
    text=[f'{v:.0f}' for v in h2h_idx['index']], textposition='bottom center'), secondary_y=True)
fig_h2h.add_hline(y=100, line_dash='dot', line_color='#BDC3C7', line_width=1.5, secondary_y=True)
fig_h2h.update_layout(title='Head-to-Head: Sub-brand Trial by Size', barmode='group',
    template='plotly_white', height=550, xaxis=dict(categoryorder='array', categoryarray=h2h_sizes))
fig_h2h.update_yaxes(title_text='Trial Shoppers', secondary_y=False)
fig_h2h.update_yaxes(title_text='Index', secondary_y=True, showgrid=False)
fig_h2h.show()

h2h_idx.to_csv(OUTPUT_DIR / 'pane_b_h2h_trial.csv', index=False)


In [ ]:
# ── C-3b: Size Migration Direction — Same / Size Down / Size Up ──────
# Stacked bar comparing Bold Gel Ball vs Ariel Gel Ball
# X = trial entry size, Y = % of repeat shoppers by migration direction

def classify_migration(trial_size, repeat_size, size_order):
    """Return migration direction based on SIZE_ORDER index."""
    if trial_size not in size_order or repeat_size not in size_order:
        return 'Other'
    ti, ri = size_order.index(trial_size), size_order.index(repeat_size)
    if ti == ri:
        return '① Same Size'
    elif ri > ti:
        return '③ Size Up (Larger)'
    else:
        return '② Size Down (Smaller)'

MIGRATE_ORDER  = ['① Same Size', '② Size Down (Smaller)', '③ Size Up (Larger)']
MIGRATE_COLORS = {
    '① Same Size':           '#4ECDC4',
    '② Size Down (Smaller)': '#E74C3C',
    '③ Size Up (Larger)':    '#2ECC71',
}

dfs_mig = []
for brand in [Brand_1, Brand_2]:
    repeat_shoppers_mig = df_cohort[
        (df_cohort['outcome']=='Repeat') & (df_cohort['sub_brand']==brand) &
        (~df_cohort['trial_size'].isin(EXCLUDED_SIZES)) &
        (~df_cohort['repeat_size'].isin(EXCLUDED_SIZES))
    ].copy()
    if len(repeat_shoppers_mig) == 0:
        continue
    repeat_shoppers_mig['migration_type'] = repeat_shoppers_mig.apply(
        lambda row: classify_migration(row['trial_size'], row['repeat_size'], SIZE_ORDER), axis=1
    )
    tmp_agg = repeat_shoppers_mig.groupby(['trial_size', 'migration_type']).agg(
        shoppers=('shopper_key', 'nunique')
    ).reset_index()
    tmp_agg['brand'] = brand
    dfs_mig.append(tmp_agg)

if dfs_mig:
    mig_compare = pd.concat(dfs_mig, ignore_index=True)
    mig_compare = mig_compare[mig_compare['migration_type'] != 'Other']

    # % within each brand × trial_size bucket
    totals = mig_compare.groupby(['brand', 'trial_size'])['shoppers'].transform('sum')
    mig_compare['pct'] = (mig_compare['shoppers'] / totals * 100).round(1)

    # Apply SIZE_ORDER ordering
    valid_sizes = order_and_filter_sizes(mig_compare['trial_size'].unique())
    mig_compare['trial_size'] = pd.Categorical(mig_compare['trial_size'], categories=valid_sizes, ordered=True)
    mig_compare = mig_compare[mig_compare['trial_size'].notna()].sort_values('trial_size')

    brands_list = [Brand_1, Brand_2]
    brands_available = [b for b in brands_list if b in mig_compare['brand'].values]

    fig_mig = make_subplots(
        rows=len(brands_available), cols=1,
        subplot_titles=[BRAND_LABEL.get(b, b) for b in brands_available],
        vertical_spacing=0.18
    )

    for row_idx, brand in enumerate(brands_available, 1):
        bd = mig_compare[mig_compare['brand'] == brand]
        for mtype in MIGRATE_ORDER:
            sub = bd[bd['migration_type'] == mtype]
            if len(sub) == 0:
                continue
            fig_mig.add_trace(go.Bar(
                x=sub['trial_size'].astype(str),
                y=sub['pct'],
                name=mtype,
                marker_color=MIGRATE_COLORS[mtype],
                text=sub['pct'].apply(lambda v: f'{v:.1f}%'),
                textposition='inside',
                insidetextanchor='middle',
                legendgroup=mtype,
                showlegend=(row_idx == 1)
            ), row=row_idx, col=1)

    fig_mig.update_layout(
        height=420 * len(brands_available),
        title_text=(
            'Size Migration Direction: Bold Gel Ball vs Ariel Gel Ball<br>'
            '<sup>% of repeat shoppers — Same size vs Size Down (smaller) vs Size Up (larger refill)</sup>'
        ),
        barmode='stack',
        template='plotly_white',
        legend=dict(orientation='h', y=-0.05)
    )
    fig_mig.update_yaxes(title_text='% of Repeat Shoppers', range=[0, 105])
    fig_mig.update_xaxes(title_text='Trial Entry Size')
    fig_mig.show()

    # ── DATA TABLE ────────────────────────────────────────────────────
    print('\n📊 DATA TABLE: Size Migration Direction Summary')
    print('=' * 100)
    summary_tbl = mig_compare.pivot_table(
        index=['brand', 'trial_size'],
        columns='migration_type',
        values=['pct', 'shoppers'],
        fill_value=0
    ).round(1)
    print(summary_tbl.to_string())
    mig_compare.to_csv(OUTPUT_DIR / 'pane_c_migration_direction.csv', index=False)
else:
    print('⚠️ No migration data available')

In [ ]:
# ── E-1: Funnel by Entry Size ─────────────────────────────────────────
df_journey = df_cohort[df_cohort['sub_brand']==Brand_1].copy()
df_journey_filt = df_journey[~df_journey['trial_size'].isin(EXCLUDED_SIZES)]

funnel_data = df_journey_filt.groupby('trial_size').agg(
    total_trial=('shopper_key','nunique'),
    repeat_shoppers=('outcome', lambda x: (x=='Repeat').sum()),
    lapse_shoppers=('outcome', lambda x: (x=='Lapse').sum()),
    avg_trial_asp=('trial_asp', 'mean'),
).reset_index()
funnel_data['repeat_rate_%'] = (funnel_data['repeat_shoppers'] / funnel_data['total_trial'] * 100).round(1)
funnel_data['lapse_rate_%'] = (funnel_data['lapse_shoppers'] / funnel_data['total_trial'] * 100).round(1)
funnel_data['_sort'] = funnel_data['trial_size'].map({s:i for i,s in enumerate(SIZE_ORDER)}).fillna(99)
funnel_data = funnel_data.sort_values('_sort').drop(columns='_sort')

print('📊 DATA TABLE: Shopper Funnel by Entry Size')
print('=' * 90)
print(funnel_data.to_string(index=False))

# Stacked bar
sizes = order_and_filter_sizes(funnel_data['trial_size'].unique())
sizes_display = list(reversed(sizes))
fd = funnel_data.set_index('trial_size')

fig = go.Figure()
fig.add_trace(go.Bar(y=sizes_display, x=fd.loc[sizes_display,'repeat_shoppers'], name='Repeat',
    orientation='h', marker_color='#2E8B57',
    text=[f"{v:,} ({r:.1f}%)" for v,r in zip(fd.loc[sizes_display,'repeat_shoppers'], fd.loc[sizes_display,'repeat_rate_%'])],
    textposition='inside'))
fig.add_trace(go.Bar(y=sizes_display, x=fd.loc[sizes_display,'lapse_shoppers'], name='Lapse',
    orientation='h', marker_color='#CC3333',
    text=[f"{v:,} ({r:.1f}%)" for v,r in zip(fd.loc[sizes_display,'lapse_shoppers'], fd.loc[sizes_display,'lapse_rate_%'])],
    textposition='inside'))
fig.update_layout(barmode='stack', title='Trial Outcome by Entry Size — Where Do We Lose Shoppers?',
    xaxis_title='Shoppers', template='plotly_white', height=400)
fig.show()

funnel_data.to_csv(OUTPUT_DIR / 'pane_e_funnel.csv', index=False)


In [ ]:
# ── Sankey — Trial Size → Repeat/Lapse ───────────
# Competitor shopper flow for comparison

atk_journey = df_cohort[
    (df_cohort['sub_brand']==Brand_2) &
    (~df_cohort['trial_size'].isin(EXCLUDED_SIZES))
].copy()

if len(atk_journey) > 0:
    atk_stage1 = atk_journey.groupby(['trial_size','outcome']).agg(
        count=('shopper_key','nunique'), avg_asp=('trial_asp','mean')).reset_index()

    atk_repeat_flow = atk_journey[atk_journey['outcome']=='Repeat'].groupby('repeat_size').agg(
        count=('shopper_key','nunique')).reset_index()
    atk_repeat_flow.columns = ['dest_label','count']
    atk_repeat_flow['dest_label'] = 'Ariel ' + atk_repeat_flow['dest_label']
    atk_repeat_flow['outcome'] = 'Repeat'

    # Lapse destinations for Ariel
    atk_lapsed_keys = atk_journey[atk_journey['outcome']=='Lapse']['shopper_key']
    atk_lapse_dest_raw = df_destination[
        (df_destination['source_brand']==Brand_2) &
        (df_destination['shopper_key'].isin(atk_lapsed_keys))
    ]
    if len(atk_lapse_dest_raw) > 0:
        atk_lapse_dest = atk_lapse_dest_raw.groupby(['next_sub_brand','next_size']).agg(
            count=('shopper_key','nunique')).reset_index()
        atk_lapse_dest['dest_label'] = atk_lapse_dest['next_sub_brand'] + ' ' + atk_lapse_dest['next_size'].fillna('')
    else:
        atk_lapse_dest = pd.DataFrame(columns=['dest_label','count'])

    total_atk_lapsed_s = atk_lapsed_keys.nunique()
    tracked_atk_s = atk_lapse_dest['count'].sum() if len(atk_lapse_dest) > 0 else 0
    cat_exit_atk = max(0, total_atk_lapsed_s - tracked_atk_s)

    atk_lapse_final = pd.concat([
        atk_lapse_dest[['dest_label','count']],
        pd.DataFrame([{'dest_label':'Category Exit', 'count': cat_exit_atk}])
    ], ignore_index=True)
    atk_lapse_final['outcome'] = 'Lapse'

    atk_stage2 = pd.concat([atk_repeat_flow, atk_lapse_final], ignore_index=True)
    atk_top = atk_stage2.nlargest(15, 'count')['dest_label'].tolist()
    atk_stage2['dest_simplified'] = atk_stage2['dest_label'].apply(lambda x: x if x in atk_top else 'Other Brands')
    atk_stage2 = atk_stage2.groupby(['outcome','dest_simplified']).agg(count=('count','sum')).reset_index()

    atk_trial_sizes = sorted(atk_journey['trial_size'].unique())
    atk_dests = sorted(atk_stage2['dest_simplified'].unique())
    atk_nodes = [f'Trial: {s}' for s in atk_trial_sizes] + ['Repeat','Lapse'] + [f'→ {d}' for d in atk_dests]
    atk_nidx = {n: i for i, n in enumerate(atk_nodes)}

    atk_src, atk_tgt, atk_val, ariel_col = [], [], [], []
    for _, row in atk_stage1.iterrows():
        atk_src.append(atk_nidx[f'Trial: {row["trial_size"]}'])
        atk_tgt.append(atk_nidx[row['outcome']])
        atk_val.append(row['count'])
        ariel_col.append('rgba(46,139,87,0.4)' if row['outcome']=='Repeat' else 'rgba(204,51,51,0.4)')
    for _, row in atk_stage2.iterrows():
        atk_src.append(atk_nidx[row['outcome']])
        atk_tgt.append(atk_nidx[f'→ {row["dest_simplified"]}'])
        atk_val.append(row['count'])
        if Brand_1 in row['dest_simplified']: ariel_col.append('rgba(30,144,255,0.4)')
        elif 'Ariel Gel Ball' in row['dest_simplified']: ariel_col.append('rgba(255,99,71,0.5)')
        elif 'Exit' in row['dest_simplified']: ariel_col.append('rgba(128,128,128,0.3)')
        else: ariel_col.append('rgba(255,165,0,0.3)')

    atk_ncol = ['#FF6347']*len(atk_trial_sizes) + ['#2E8B57','#CC3333'] + ['#6495ED']*len(atk_dests)
    fig_atk_s = go.Figure(go.Sankey(
        node=dict(pad=15, thickness=20, label=atk_nodes, color=atk_ncol),
        link=dict(source=atk_src, target=atk_tgt, value=atk_val, color=ariel_col)))
    fig_atk_s.update_layout(
        title_text='Ariel Gel Ball: Shopper Flow — Trial Size → Repeat/Lapse → Destination',
        font_size=11, height=700, template='plotly_white')
    fig_atk_s.show()

    # ── DATA TABLE ────────────────────────────────────────────────────
    print('\n📊 DATA TABLE: Ariel Gel Ball Shopper Flow — Stage 1')
    print('=' * 80)
    atk_s1_display = atk_stage1[['trial_size','outcome','count','avg_asp']].copy()
    atk_s1_display['avg_asp'] = atk_s1_display['avg_asp'].round(0)
    print(atk_s1_display.to_string(index=False))

    print('\n📊 DATA TABLE: Ariel Gel Ball Shopper Flow — Stage 2 (Destinations)')
    print('=' * 80)
    print(atk_stage2.sort_values(['outcome','count'], ascending=[True,False]).to_string(index=False))

    atk_stage1.to_csv(OUTPUT_DIR / 'pane_e_attack_sankey_stage1.csv', index=False)
    atk_stage2.to_csv(OUTPUT_DIR / 'pane_e_attack_sankey_stage2.csv', index=False)
else:
    print('⚠️ No Ariel cohort data available for Sankey')

In [ ]:
# ── Helper: Per-Size Sankey — Trial Size → Repeat/Lapse → Destination ─
def _plot_per_size_sankey(brand_code, brand_journey, df_destination):
    """Plot one standalone Sankey per entry size for the given brand."""
    dest_all = df_destination[df_destination['source_brand'] == brand_code]
    per_size_sizes = order_and_filter_sizes(brand_journey['trial_size'].unique())

    for size in per_size_sizes:
        # --- Stage 1: Trial → Repeat / Lapse ---
        size_journey = brand_journey[brand_journey['trial_size'] == size]
        s1 = size_journey.groupby('outcome').agg(
            count=('shopper_key', 'nunique'),
            avg_asp=('trial_asp', 'mean')
        ).reset_index()

        # --- Stage 2a: Repeat → destination (repeat size within same brand) ---
        rep_flow = size_journey[size_journey['outcome'] == 'Repeat'].groupby('repeat_size').agg(
            count=('shopper_key', 'nunique')
        ).reset_index()
        rep_flow.columns = ['dest_label', 'count']
        rep_flow['dest_label'] = brand_code + ' ' + rep_flow['dest_label']
        rep_flow['outcome'] = 'Repeat'

        # --- Stage 2b: Lapse → destination (next sub-brand / size from df_destination) ---
        lapsed_keys = size_journey[size_journey['outcome'] == 'Lapse']['shopper_key']
        lapse_raw = dest_all[dest_all['shopper_key'].isin(lapsed_keys)]
        if len(lapse_raw) > 0:
            lapse_dest = lapse_raw.groupby(['next_sub_brand', 'next_size']).agg(
                count=('shopper_key', 'nunique')
            ).reset_index()
            lapse_dest['dest_label'] = lapse_dest['next_sub_brand'] + ' ' + lapse_dest['next_size'].fillna('')
        else:
            lapse_dest = pd.DataFrame(columns=['dest_label', 'count'])

        total_lapsed_s = lapsed_keys.nunique()
        tracked_lapsed = lapse_dest['count'].sum() if len(lapse_dest) > 0 else 0
        cat_exit_s = max(0, total_lapsed_s - tracked_lapsed)
        lapse_final = pd.concat([
            lapse_dest[['dest_label', 'count']],
            pd.DataFrame([{'dest_label': 'Category Exit', 'count': cat_exit_s}])
        ], ignore_index=True)
        lapse_final['outcome'] = 'Lapse'

        # --- Combine stage 2 and simplify ---
        s2 = pd.concat([rep_flow, lapse_final], ignore_index=True)
        s2['count'] = pd.to_numeric(s2['count'], errors='coerce').fillna(0).astype(int)
        top_d = s2.nlargest(12, 'count')['dest_label'].tolist()
        s2['dest_simplified'] = s2['dest_label'].apply(lambda x: x if x in top_d else 'Other Brands')
        s2 = s2.groupby(['outcome', 'dest_simplified']).agg(count=('count', 'sum')).reset_index()

        # --- Build Sankey nodes & links ---
        nodes_list = [f'Trial: {size}'] + ['Repeat', 'Lapse'] + [f'→ {d}' for d in sorted(s2['dest_simplified'].unique())]
        nidx = {n: i for i, n in enumerate(nodes_list)}

        sources, targets, values, colors = [], [], [], []
        for _, row in s1.iterrows():
            sources.append(nidx[f'Trial: {size}'])
            targets.append(nidx[row['outcome']])
            values.append(row['count'])
            colors.append('rgba(46,139,87,0.4)' if row['outcome'] == 'Repeat' else 'rgba(204,51,51,0.4)')
        for _, row in s2.iterrows():
            sources.append(nidx[row['outcome']])
            targets.append(nidx[f'→ {row["dest_simplified"]}'])
            values.append(row['count'])
            if brand_code in row['dest_simplified']:
                colors.append('rgba(30,144,255,0.4)')
            elif Brand_2 in row['dest_simplified']:
                colors.append('rgba(255,99,71,0.5)')
            elif 'Exit' in row['dest_simplified']:
                colors.append('rgba(128,128,128,0.3)')
            else:
                colors.append('rgba(255,165,0,0.3)')

        n_dests = len(sorted(s2['dest_simplified'].unique()))
        node_colors = ['#1E90FF'] + ['#2E8B57', '#CC3333'] + ['#6495ED'] * n_dests

        fig_s = go.Figure(go.Sankey(
            node=dict(pad=15, thickness=20, label=nodes_list, color=node_colors),
            link=dict(source=sources, target=targets, value=values, color=colors)
        ))
        total_s = s1['count'].sum()
        repeat_s = s1.loc[s1['outcome'] == 'Repeat', 'count'].sum() if 'Repeat' in s1['outcome'].values else 0
        rr = round(repeat_s / max(total_s, 1) * 100, 1)
        fig_s.update_layout(
            title_text=(f'【{size}】Shopper Flow: Trial → Repeat/Lapse → Destination<br>'
                        f'<sup>Total trial: {total_s:,} | Repeat: {repeat_s:,} ({rr}%) | Lapse: {total_s - repeat_s:,} ({round(100-rr,1)}%)</sup>'),
            font_size=11, height=600, template='plotly_white'
        )
        fig_s.show()

        # Data table
        print(f'\n📊 {size} — Stage 1')
        print(s1.to_string(index=False))
        print(f'\n📊 {size} — Stage 2 (Top destinations)')
        print(s2.sort_values(['outcome', 'count'], ascending=[True, False]).to_string(index=False))


# ── E-2b: Per-Size Sankey — Brand_1 (Trial Size → Repeat/Lapse → Destination) ─
_plot_per_size_sankey(Brand_1, df_journey, df_destination)

In [ ]:
# ── E-2b: Per-Size Sankey — Brand_2 (Trial Size → Repeat/Lapse → Destination) ─
_plot_per_size_sankey(Brand_2, atk_journey, df_destination)

In [ ]:
# ── E-2: Sankey — Trial Size → Repeat/Lapse → Destination ────────────
flow_stage1 = df_journey.groupby(['trial_size','outcome']).agg(
    count=('shopper_key','nunique'), avg_asp=('trial_asp','mean')).reset_index()

repeat_flow = df_journey[df_journey['outcome']=='Repeat'].groupby('repeat_size').agg(
    count=('shopper_key','nunique')).reset_index()
repeat_flow.columns = ['dest_label', 'count']
repeat_flow['dest_label'] = 'Bold ' + repeat_flow['dest_label']
repeat_flow['outcome'] = 'Repeat'

lapse_dest = dest_summary.copy()
lapse_dest['dest_label'] = lapse_dest['next_sub_brand'] + ' ' + lapse_dest['next_size'].fillna('')
lapse_dest = lapse_dest.rename(columns={'shoppers':'count'})
total_lapsed_j = (df_journey['outcome']=='Lapse').sum()
cat_exit = max(0, total_lapsed_j - lapse_dest['count'].sum())
lapse_dest = pd.concat([lapse_dest[['dest_label','count']],
    pd.DataFrame([{'dest_label':'Category Exit', 'count':cat_exit}])], ignore_index=True)
lapse_dest['outcome'] = 'Lapse'

flow_stage2 = pd.concat([repeat_flow, lapse_dest], ignore_index=True)
top_dests = flow_stage2.nlargest(15, 'count')['dest_label'].tolist()
flow_stage2['dest_simplified'] = flow_stage2['dest_label'].apply(lambda x: x if x in top_dests else 'Other Brands')
flow_stage2 = flow_stage2.groupby(['outcome','dest_simplified']).agg(count=('count','sum')).reset_index()

trial_sizes = sorted(df_journey['trial_size'].unique())
destinations = sorted(flow_stage2['dest_simplified'].unique())
all_nodes = [f'Trial: {s}' for s in trial_sizes] + ['Repeat','Lapse'] + [f'→ {d}' for d in destinations]
node_idx = {n:i for i,n in enumerate(all_nodes)}

sources, targets, values, colors = [], [], [], []
for _, row in flow_stage1.iterrows():
    sources.append(node_idx[f'Trial: {row["trial_size"]}'])
    targets.append(node_idx[row['outcome']])
    values.append(row['count'])
    colors.append('rgba(46,139,87,0.4)' if row['outcome']=='Repeat' else 'rgba(204,51,51,0.4)')
for _, row in flow_stage2.iterrows():
    sources.append(node_idx[row['outcome']])
    targets.append(node_idx[f'→ {row["dest_simplified"]}'])
    values.append(row['count'])
    if 'Bold' in row['dest_simplified']: colors.append('rgba(30,144,255,0.4)')
    elif Brand_2 in row['dest_simplified']: colors.append('rgba(255,99,71,0.5)')
    elif 'Exit' in row['dest_simplified']: colors.append('rgba(128,128,128,0.3)')
    else: colors.append('rgba(255,165,0,0.3)')

node_colors = ['#1E90FF']*len(trial_sizes) + ['#2E8B57','#CC3333'] + ['#6495ED']*len(destinations)
fig = go.Figure(go.Sankey(
    node=dict(pad=15, thickness=20, label=all_nodes, color=node_colors),
    link=dict(source=sources, target=targets, value=values, color=colors)))
fig.update_layout(title_text='Shopper Flow: Trial Size → Repeat/Lapse → Destination', font_size=11, height=700, template='plotly_white')
fig.show()

# ── DATA TABLE: Sankey Flow Summary ──────────────────────────────────
print('\n📊 DATA TABLE: Shopper Flow — Stage 1 (Trial Size → Outcome)')
print('=' * 80)
fs1_display = flow_stage1[['trial_size','outcome','count','avg_asp']].copy()
fs1_display['avg_asp'] = fs1_display['avg_asp'].round(0)
print(fs1_display.to_string(index=False))

print('\n📊 DATA TABLE: Shopper Flow — Stage 2 (Outcome → Destination)')
print('=' * 80)
fs2_display = flow_stage2[['outcome','dest_simplified','count']].sort_values(['outcome','count'], ascending=[True,False])
print(fs2_display.to_string(index=False))

flow_stage1.to_csv(OUTPUT_DIR / 'pane_e_sankey_stage1.csv', index=False)
flow_stage2.to_csv(OUTPUT_DIR / 'pane_e_sankey_stage2.csv', index=False)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Multi-Size Price Point Decision Matrix
# ═══════════════════════════════════════════════════════════════════════
#   ▶ Freq(w×s)  : uses df_asp_weekly_store with store-level ASP band grain.
#   ▶ avg_asp    : computed from asp_band midpoints (store-level), not weighted_asp averages.


# ── Recompute market_freq from store-level ASP bands (Q6b) ────────────
market_freq_v2 = (
    df_asp_weekly_store[df_asp_weekly_store['sub_brand'] == Brand_1]
    .groupby(['size_code', 'asp_band'], as_index=False)
    .agg(store_exec_freq=('site_key', 'nunique'))   # distinct stores, not sum of n_stores
    .rename(columns={'size_code': 'trial_size'})
)

# ── Recompute bold_avg_asp from store-level asp_band midpoints ─────────
bold_avg_asp_by_size_v2 = (
    df_asp_weekly_store[
        (df_asp_weekly_store['sub_brand'] == Brand_1) &
        (~df_asp_weekly_store['size_code'].isin(EXCLUDED_SIZES))
    ]
    .assign(asp_mid=lambda d: d['asp_band'] + 25)
    .groupby('size_code')
    .agg(avg_asp=('asp_mid', 'mean'), median_asp=('asp_mid', 'median'))
    .reset_index()
)

# ── Revised build_size_matrix (v2) ────────────────────────────────────
def build_size_matrix_v2(target_size):
    sz_cohort = df_cohort[
        (df_cohort['sub_brand'] == Brand_1) &
        (df_cohort['trial_size'] == target_size) &
        (df_cohort['trial_asp'].notna())
    ].copy()
    if len(sz_cohort) == 0:
        return pd.DataFrame()

    sz_cohort['asp_band'] = (sz_cohort['trial_asp'] // 50 * 50).astype('Int64')

    outcome_agg = sz_cohort.groupby(['asp_band', 'outcome']).agg(
        shoppers=('shopper_key', 'nunique')
    ).reset_index()

    base = outcome_agg.pivot_table(
        index='asp_band', columns='outcome', values='shoppers', fill_value=0
    ).reset_index()
    if 'Repeat' not in base.columns: base['Repeat'] = 0
    if 'Lapse'  not in base.columns: base['Lapse']  = 0
    base['total'] = base['Repeat'] + base['Lapse']
    base = base[base['total'] > 0].sort_values('asp_band').reset_index(drop=True)
    if len(base) == 0:
        return pd.DataFrame()

    mig  = _migration_by_band(target_size)
    dest = _lapse_dest_by_band(target_size)

    # ▶ KEY FIX: store-level freq from market_freq_v2
    freq_map = (
        market_freq_v2[market_freq_v2['trial_size'] == target_size]
        .set_index('asp_band')['store_exec_freq']
        .to_dict()
    )

    univ_sub = df_universe[df_universe['trial_size'] == target_size]
    univ_map = univ_sub.set_index('asp_band')['all_shoppers'].to_dict()

    sz_idx   = SIZE_ORDER.index(target_size) if target_size in SIZE_ORDER else -1
    next_size = SIZE_ORDER[sz_idx + 1] if 0 <= sz_idx < len(SIZE_ORDER) - 1 else None
    next_avg  = None
    if next_size is not None:
        nrow = bold_avg_asp_by_size_v2[bold_avg_asp_by_size_v2['size_code'] == next_size]
        if len(nrow) > 0:
            next_avg = float(nrow['avg_asp'].values[0])

    rows = []
    for _, b in base.iterrows():
        bv    = int(b['asp_band'])
        total = int(b['total'])
        rep_n = int(b['Repeat'])
        lap_n = int(b['Lapse'])
        freq  = int(freq_map.get(bv, 0))

        repeat_pct = round(rep_n / total * 100, 1) if total > 0 else 0.0
        lapse_pct  = round(lap_n / total * 100, 1) if total > 0 else 0.0
        trial_eff  = round(total / freq  * 100, 1) if freq  > 0 else None

        m   = mig[mig['asp_band']   == bv]
        d   = dest[dest['asp_band'] == bv]
        gap = round(next_avg - (bv + 25), 0) if next_avg is not None else None
        total_n = int(univ_map.get(bv, 0))

        rows.append({
            'size':               target_size,
            'asp_band':           bv,
            'price_band':         f'~¥{bv + 49:,}',
            'frequency':          freq,
            'trial_efficiency_%': trial_eff,
            'Total N':            total_n,
            'Trial N':            total,
            'repeat_%':           repeat_pct,
            'lapse_%':            lapse_pct,
            'same_size_%':        m['same_pct'].values[0] if len(m) > 0 else 0.0,
            'size_up_%':          m['up_pct'].values[0]   if len(m) > 0 else 0.0,
            'size_dn_%':          m['dn_pct'].values[0]   if len(m) > 0 else 0.0,
            '→Ariel%':            d['ariel_pct'].values[0] if len(d) > 0 else 0.0,
            '→Exit%':             d['exit_pct'].values[0]  if len(d) > 0 else 0.0,
            'gap_to_next_jpy':    gap,
            '_freq_flag':         '*' if freq < MIN_FREQ_FLAG_G else '',
        })

    return pd.DataFrame(rows)

# ── Build & print all sizes (revised) ────────────────────────────────
all_size_matrices_v2 = {}
for size in SIZE_ORDER:
    mat = build_size_matrix_v2(size)
    if len(mat) == 0:
        print(f'  ⚠️ No data for {size} — skipped')
        continue
    all_size_matrices_v2[size] = mat

    abbr     = _size_abbr(size)
    next_lbl = _NEXT_SIZE_LBL.get(size) or '(top of ladder)'
    role     = SIZE_ROLE.get(size, '?')

    print()
    print('=' * 165)
    print(f'📊 G-1 (REVISED): {size}  [Freq = store-level ASP band grain — Q6b]')
    print(f'  Role: {role}  |  Next size → {next_lbl}  |  * = freq < {MIN_FREQ_FLAG_G:,}')
    print(f'  FIX: Freq(w×s) uses Q6b store-level bands — weighted_asp inflation removed')
    print('=' * 165)
    print(f'{"Price Band":>12s} {"Freq(w×s)":>10s} {"TrialEff%":>10s} {"Total N":>9s} {"Trial N":>8s} '
          f'{"Repeat%":>8s} {"Lapse%":>8s} '
          f'{"Same"+abbr+"%":>9s} {"SizeUp%":>8s} {"SizeDn%":>8s} '
          f'{"→Ariel%":>8s} {"→Exit%":>8s} {"Gap→Next":>9s}')
    print('-' * 165)
    for _, r in mat.iterrows():
        gap_s = f'¥{r["gap_to_next_jpy"]:+,.0f}' if pd.notna(r.get('gap_to_next_jpy')) else '—'
        eff_s = f'{r["trial_efficiency_%"]:.1f}%'  if pd.notna(r.get('trial_efficiency_%')) else '—'
        bl    = f'{r["price_band"]}{r["_freq_flag"]}'
        print(f'{bl:>13s} {r["frequency"]:>10,} {eff_s:>10s} {r["Total N"]:>9,} {r["Trial N"]:>8,} '
              f'{r["repeat_%"]:>7.1f}% {r["lapse_%"]:>7.1f}% '
              f'{r["same_size_%"]:>8.1f}% {r["size_up_%"]:>7.1f}% {r["size_dn_%"]:>7.1f}% '
              f'{r["→Ariel%"]:>7.1f}% {r["→Exit%"]:>7.1f}% {gap_s:>9s}')

print(f'\n✅ G-1 (REVISED) complete — {list(all_size_matrices_v2.keys())}')


In [2]:
# ── Export: All pulled and derived data to Excel ───────────────────────
# Exports raw query results and all derived analysis tables as a reference.
output_path = OUTPUT_DIR / 'trial_repeat_lapse_reference.xlsx'

_sheets = {}

# ── Raw query results ──────────────────────────────────────────────────
_sheets['Q1_Cohort'] = strip_tz(df_cohort)
_sheets['Q2_Destination'] = strip_tz(df_destination)

# ── Derived analysis tables (written only if the variable exists) ──────
for _var, _sheet in [
    ('h2h_idx',     'B_H2H_Trial'),
    ('mig_compare', 'C_Migration_Dir'),
    ('funnel_data', 'E_Funnel'),
    ('flow_stage1', 'E_Sankey_Stage1'),
    ('flow_stage2', 'E_Sankey_Stage2'),
]:
    _obj = globals().get(_var)
    if isinstance(_obj, pd.DataFrame) and len(_obj) > 0:
        _sheets[_sheet] = strip_tz(_obj)

# ── Price Point Matrix (all sizes stacked vertically) ─────────────────
if 'all_size_matrices_v2' in globals() and len(all_size_matrices_v2) > 0:
    _export_cols = [
        'size', 'price_band', 'frequency', 'trial_efficiency_%',
        'Total N', 'Trial N', 'repeat_%', 'lapse_%',
        'same_size_%', 'size_up_%', 'size_dn_%',
        '→Ariel%', '→Exit%', 'gap_to_next_jpy',
    ]
    _parts = []
    for _sz, _mat in all_size_matrices_v2.items():
        _sep = pd.DataFrame([{c: '' for c in _export_cols}])
        _sep.at[0, 'size'] = f'▼ {_sz}'
        _avail = [c for c in _export_cols if c in _mat.columns]
        _me = _mat[_avail].copy()
        for c in _export_cols:
            if c not in _me.columns:
                _me[c] = ''
        _parts.extend([_sep, _me[_export_cols]])
    _sheets['G_PriceMatrix'] = strip_tz(pd.concat(_parts, ignore_index=True))

with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    for _sheet_name, _df in _sheets.items():
        _df.to_excel(writer, sheet_name=_sheet_name, index=False)

print(f'✅ Reference export saved → {output_path}')
for _sh in _sheets:
    print(f'   • {_sh}  ({len(_sheets[_sh]):,} rows)')